# Taller Final Minería de Datos: Segmentación de clientes con K-Means

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
sns.set_theme(style="whitegrid")

## Parte 1 – Exploración de datos

In [ ]:
url = "https://raw.githubusercontent.com/SteffiPeTaffy/machineLearningAZ/master/Machine%20Learning%20A-Z%20Template%20Folder/Part%204%20-%20Clustering/Section%2025%20-%20Hierarchical%20Clustering/Mall_Customers.csv"
try:
    raw = pd.read_csv(url)
except Exception:
    from google.colab import files
    files.upload()
    raw = pd.read_csv("Mall_Customers.csv")
raw.to_csv("Mall_Customers.csv", index=False)

df = raw.rename(columns={"Genre": "Gender", "Annual Income (k$)": "Income", "Spending Score (1-100)": "Spending"})
print(df.shape)
df.head()

In [ ]:
df.info()
print(df.isnull().sum())
print("Duplicados:", df.duplicated().sum())
df["Gender"].value_counts()

In [ ]:
df.describe().round(2)

In [ ]:
num_cols = ["Age", "Income", "Spending"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, col in enumerate(num_cols):
    sns.histplot(df[col], kde=True, ax=axes[0, i])
    sns.boxplot(y=df[col], ax=axes[1, i], color="lightcoral")
plt.tight_layout(); plt.show()

In [ ]:
for col in num_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    out = df[(df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)][col]
    print(col, len(out), out.tolist())

In [ ]:
sns.pairplot(df, vars=num_cols, hue="Gender")
plt.show()
sns.heatmap(df[num_cols].corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.show()

**¿Qué variables son más adecuadas para segmentar clientes?**
Las variables que mejor sirven son el ingreso anual y el puntaje de gasto, porque muestran cuánto gana cada cliente y cuánto gasta en el centro comercial. El CustomerID no sirve porque solo es un número para identificar a cada persona, y el género lo dejamos por fuera porque es una variable de texto.

**¿Se identifican valores atípicos?**
Casi no hay. En los diagramas de caja solo aparecen 2 clientes con un ingreso de 137 k$, que es bastante más alto que el resto. No los eliminamos porque no son errores, simplemente son clientes que ganan mucho.

**¿Qué tipo de relaciones se observan entre las variables?**
Entre el ingreso y el gasto no se ve una relación directa, o sea, ganar más no significa gastar más. Pero en la gráfica sí se ven grupos de clientes bien separados. También notamos que la edad y el gasto tienen una relación negativa: los clientes más jóvenes suelen gastar más.

## Parte 2 – Preprocesamiento

In [ ]:
features = ["Income", "Spending"]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])
print(X_scaled.mean(axis=0).round(4), X_scaled.std(axis=0).round(4))

**¿Por qué es necesario escalar las variables en el algoritmo K-Means?**
Porque K-Means agrupa a los clientes según qué tan cerca están unos de otros. Si una variable tiene números mucho más grandes que la otra, termina pesando más y el resultado sale sesgado. Al escalar, las dos variables quedan en la misma escala y cuentan por igual.

## Parte 3 – Método del codo

In [ ]:
inertias = []
for k in range(1, 11):
    inertias.append(KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_scaled).inertia_)

plt.plot(range(1, 11), inertias, "o-")
plt.axvline(5, color="red", linestyle="--")
plt.xlabel("K"); plt.ylabel("Inercia (WCSS)"); plt.title("Método del codo")
plt.show()
pd.DataFrame({"K": range(1, 11), "Inercia": np.round(inertias, 2)})

**¿Por qué valores cercanos a cero indican mejor ajuste?**
Porque la inercia mide qué tan lejos están los clientes del centro de su grupo. Si es cercana a cero, quiere decir que los clientes de cada grupo se parecen mucho entre sí. Eso sí, la inercia siempre baja cuando se agregan más grupos, por eso no se busca el valor más bajo sino el codo.

**¿Cómo se identifica el punto óptimo de K?**
Se mira en la gráfica el punto donde la línea deja de bajar tan rápido y se empieza a aplanar, como si fuera un codo. En nuestro caso eso pasa en K = 5, porque después de ahí agregar más grupos casi no mejora el resultado.

## Parte 4 – Coeficiente de silueta

In [ ]:
sil = []
for k in range(2, 11):
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_scaled)
    sil.append(silhouette_score(X_scaled, labels))

plt.plot(range(2, 11), sil, "o-", color="seagreen")
plt.xlabel("K"); plt.ylabel("Silueta"); plt.title("Coeficiente de silueta")
plt.show()
sil_df = pd.DataFrame({"K": range(2, 11), "Silueta": np.round(sil, 4)})
print("Mejor K:", sil_df.loc[sil_df["Silueta"].idxmax(), "K"])
sil_df

**¿Qué mide el coeficiente de silueta?**
Mide qué tan bien quedó cada cliente en su grupo. Compara qué tan cerca está de los clientes de su mismo grupo con qué tan cerca está del grupo vecino. Su valor va de -1 a 1.

**¿Qué significa un valor cercano a 1?**
Que los grupos están bien formados: los clientes de cada grupo están juntos y bien separados de los otros grupos.

**¿Qué implica un valor cercano a 0 o negativo?**
Si es cercano a 0, el cliente está en el límite entre dos grupos y no queda claro a cuál pertenece. Si es negativo, es probable que el cliente haya quedado en el grupo equivocado.

**¿Coincide el valor óptimo de K con el método del codo? Justifique.**
Sí coincide. El valor más alto de la silueta fue con K = 5 (0.555), que es el mismo número que nos dio el método del codo. Como los dos métodos llegan al mismo resultado, podemos estar seguros de que 5 grupos es una buena elección.

## Parte 5 – Modelo K-Means

In [ ]:
kmeans = KMeans(n_clusters=5, n_init=10, random_state=42)
df["Cluster"] = kmeans.fit_predict(X_scaled)

centroids = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=features)
centroids.round(2)

In [ ]:
sns.scatterplot(data=df, x="Income", y="Spending", hue="Cluster", palette="tab10", s=60)
plt.scatter(centroids["Income"], centroids["Spending"], c="black", s=250, marker="X")
plt.title("Clusters (K = 5)")
plt.show()

df.groupby("Cluster").agg(Clientes=("CustomerID", "count"), Edad=("Age", "mean"),
                          Ingreso=("Income", "mean"), Gasto=("Spending", "mean")).round(1)

**¿Qué representa cada cluster?**
Cada cluster es un grupo de clientes que se parecen en cuánto ganan y cuánto gastan. El centroide sería como el cliente promedio de ese grupo. Nos salieron 5 grupos: los que ganan y gastan mucho, los que están en el promedio, los que ganan poco pero gastan mucho, los que ganan mucho pero gastan poco, y los que ganan y gastan poco.

**¿Qué diferencias existen entre los grupos obtenidos?**
La principal diferencia es cómo gastan según lo que ganan. Por ejemplo, hay dos grupos que ganan casi lo mismo, pero uno gasta mucho y el otro casi nada. También vimos que los grupos que más gastan son los más jóvenes (entre 25 y 33 años en promedio). El grupo promedio es el más grande, con 81 clientes.

## Parte 6 – Interpretación de negocio

In [ ]:
def perfil(inc, spe):
    im, isd = df["Income"].mean(), df["Income"].std()
    sm, ssd = df["Spending"].mean(), df["Spending"].std()
    alto_i, bajo_i = inc > im + 0.5 * isd, inc < im - 0.5 * isd
    alto_g, bajo_g = spe > sm + 0.5 * ssd, spe < sm - 0.5 * ssd
    if alto_i and alto_g: return "Clientes de alto valor"
    if alto_i and bajo_g: return "Clientes con potencial"
    if bajo_i and bajo_g: return "Clientes de bajo valor"
    if bajo_i and alto_g: return "Clientes impulsivos (gasto alto, ingreso bajo)"
    return "Clientes promedio"

centroids["Perfil"] = [perfil(r.Income, r.Spending) for r in centroids.itertuples()]
df["Perfil"] = df["Cluster"].map(centroids["Perfil"])

sns.scatterplot(data=df, x="Income", y="Spending", hue="Perfil", palette="Set1", s=60)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()
centroids.round(1)

**¿Qué estrategias de marketing se podrían aplicar a cada grupo?**

- **Alto valor:** darles beneficios especiales para que sigan comprando, como una tarjeta VIP o descuentos exclusivos.
- **Promedio:** hacer promociones y programas de puntos para que vengan más seguido.
- **Con potencial:** enviarles ofertas personalizadas y productos de mejor calidad para que se animen a comprar más.
- **Bajo valor:** ofrecerles descuentos y promociones de temporada sin gastar mucho en publicidad.
- **Impulsivos:** llegarles por redes sociales con productos de moda y opciones de pago a cuotas.

**¿Qué segmento representa mayor oportunidad de crecimiento?**
Los clientes con potencial, porque ganan bien (alrededor de 88 k$) pero gastan muy poco (puntaje de 17). Ya tienen el dinero, solo hace falta encontrar la forma de que compren más en el centro comercial.

## Parte 7 – Despliegue con Streamlit

In [ ]:
!pip install -q streamlit

In [ ]:
%%writefile app.py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import streamlit as st
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

st.set_page_config(page_title="Segmentación de clientes", page_icon="🛍️", layout="wide")
FEATURES = ["Income", "Spending"]
RANDOM_STATE = 42

ESTRATEGIAS = {
    "Clientes de alto valor": "Fidelización VIP, beneficios exclusivos, productos premium y venta cruzada.",
    "Clientes promedio": "Programa de puntos, combos y promociones periódicas para subir la frecuencia de compra.",
    "Clientes con potencial": "Campañas personalizadas, experiencias exclusivas y marcas de mayor calidad para activar su gasto.",
    "Clientes de bajo valor": "Comunicación de bajo costo y ofertas de precio en temporadas de descuento.",
    "Clientes impulsivos (gasto alto, ingreso bajo)": "Promociones de tendencia, redes sociales, pagos a cuotas y descuentos para estudiantes.",
}


@st.cache_data
def cargar_datos():
    df = pd.read_csv("Mall_Customers.csv")
    return df.rename(columns={"Genre": "Gender",
                              "Annual Income (k$)": "Income",
                              "Spending Score (1-100)": "Spending"})


@st.cache_data
def metricas_k(X_scaled):
    filas = []
    for k in range(1, 11):
        km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_scaled)
        sil = silhouette_score(X_scaled, km.labels_) if k > 1 else np.nan
        filas.append({"K": k, "Inercia": km.inertia_, "Silueta": sil})
    return pd.DataFrame(filas)


def perfil(income, spending, df, tol=0.5):
    im, isd = df["Income"].mean(), df["Income"].std()
    sm, ssd = df["Spending"].mean(), df["Spending"].std()
    ai, bi = income > im + tol * isd, income < im - tol * isd
    ag, bg = spending > sm + tol * ssd, spending < sm - tol * ssd
    if ai and ag: return "Clientes de alto valor"
    if ai and bg: return "Clientes con potencial"
    if bi and bg: return "Clientes de bajo valor"
    if bi and ag: return "Clientes impulsivos (gasto alto, ingreso bajo)"
    return "Clientes promedio"


df = cargar_datos()
scaler = StandardScaler().fit(df[FEATURES])
X_scaled = scaler.transform(df[FEATURES])
met = metricas_k(X_scaled)

# ---------------- Barra lateral
st.sidebar.header("⚙️ Configuración")
k = st.sidebar.slider("Número de clusters (K)", 2, 10, 5)
st.sidebar.caption("K = 5 es el óptimo según el codo y la silueta.")

kmeans = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_scaled)
df["Cluster"] = kmeans.labels_
centroides = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=FEATURES)
centroides["Perfil"] = [perfil(r.Income, r.Spending, df) for r in centroides.itertuples()]
df["Perfil"] = df["Cluster"].map(centroides["Perfil"])

st.title("🛍️ Segmentación de clientes con K-Means")
st.write("Mall Customers Dataset · variables: **ingreso anual (k$)** y **puntaje de gasto (1-100)**")

c1, c2, c3 = st.columns(3)
c1.metric("Clientes", len(df))
c2.metric("Clusters (K)", k)
c3.metric("Coeficiente de silueta", f"{silhouette_score(X_scaled, kmeans.labels_):.3f}")

tab1, tab2, tab3 = st.tabs(["📉 Método del codo", "🎯 Clusters", "➕ Nuevo cliente"])

# ---------------- Tab 1: codo
with tab1:
    col_a, col_b = st.columns(2)
    with col_a:
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.plot(met["K"], met["Inercia"], "o-", color="steelblue", lw=2)
        ax.axvline(k, color="red", ls="--", label=f"K seleccionado = {k}")
        ax.set_xlabel("K"); ax.set_ylabel("Inercia (WCSS)"); ax.set_title("Método del codo")
        ax.set_xticks(range(1, 11)); ax.legend()
        st.pyplot(fig)
    with col_b:
        fig, ax = plt.subplots(figsize=(6, 4))
        m2 = met.dropna()
        ax.plot(m2["K"], m2["Silueta"], "o-", color="seagreen", lw=2)
        ax.axvline(k, color="red", ls="--", label=f"K seleccionado = {k}")
        ax.set_xlabel("K"); ax.set_ylabel("Silueta"); ax.set_title("Coeficiente de silueta")
        ax.set_xticks(range(2, 11)); ax.legend()
        st.pyplot(fig)
    st.dataframe(met.round(3), hide_index=True, width="stretch")

# ---------------- Tab 2: clusters
with tab2:
    fig, ax = plt.subplots(figsize=(9, 5.5))
    sns.scatterplot(data=df, x="Income", y="Spending", hue="Perfil", style="Cluster",
                    palette="Set1", s=70, ax=ax)
    ax.scatter(centroides["Income"], centroides["Spending"], c="black", s=250, marker="X", label="Centroides")
    ax.set_xlabel("Ingreso anual (k$)"); ax.set_ylabel("Puntaje de gasto (1-100)")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    st.pyplot(fig)

    resumen = df.groupby(["Cluster", "Perfil"]).agg(
        Clientes=("CustomerID", "count"), Edad_media=("Age", "mean"),
        Ingreso_medio=("Income", "mean"), Gasto_medio=("Spending", "mean")).round(1).reset_index()
    st.subheader("Resumen por cluster")
    st.dataframe(resumen, hide_index=True, width="stretch")

# ---------------- Tab 3: nuevo cliente
with tab3:
    st.subheader("Clasificar un nuevo cliente")
    with st.form("nuevo"):
        a, b = st.columns(2)
        ingreso = a.number_input("Ingreso anual (k$)", min_value=0.0, max_value=300.0, value=70.0, step=1.0)
        gasto = b.slider("Puntaje de gasto (1-100)", 1, 100, 50)
        ok = st.form_submit_button("Clasificar")

    if ok:
        nuevo = pd.DataFrame([[ingreso, gasto]], columns=FEATURES)
        cl = int(kmeans.predict(scaler.transform(nuevo))[0])
        pf = centroides.loc[cl, "Perfil"]
        st.success(f"El cliente pertenece al **Cluster {cl}** → **{pf}**")
        st.info(f"💡 Estrategia sugerida: {ESTRATEGIAS[pf]}")

        fig, ax = plt.subplots(figsize=(9, 5.5))
        sns.scatterplot(data=df, x="Income", y="Spending", hue="Perfil", palette="Set1",
                        s=50, alpha=0.5, ax=ax)
        ax.scatter(centroides["Income"], centroides["Spending"], c="black", s=200, marker="X")
        ax.scatter(ingreso, gasto, c="gold", edgecolor="black", s=400, marker="*", label="Nuevo cliente", zorder=5)
        ax.set_xlabel("Ingreso anual (k$)"); ax.set_ylabel("Puntaje de gasto (1-100)")
        ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
        st.pyplot(fig)

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!streamlit run app.py --server.headless true &>/content/streamlit.log &
!./cloudflared tunnel --url http://localhost:8501